# Stage-1 test of the CodeBERT in `models/` on real projects and on PetClinic

**The question.** The app now ranks code with three signals: TF-IDF + MiniLM + the fine-tuned CodeBERT in `models/c4-codebert`. Is that better than the two signals it used before? This notebook measures it on the **seven real, expert-labelled projects** and on **Spring PetClinic**, with graphs and a statistical check.

**Everything runs on this computer** (no Neo4j, no LLM, no internet), in a few minutes. It uses the project's own retrieval code (`retrieval.py`), so the numbers are what the app really produces.

**Honest limit.** This model was fine-tuned on other (LLM-written) data and never saw these projects, so these are fair tests of whether it transfers. The seven projects are small and differ a lot; the bootstrap check in section 5 says how much of a difference could be luck.

## 1. Environment

This cell works on your Mac and in Google Colab. On Colab it installs what is missing and asks you to upload `colab_bundle.zip` (the small project files and datasets this notebook needs). On your Mac it just finds the project folder. It also picks the fastest device it can find: a GPU (`cuda` on Colab, `mps` on a Mac) or the CPU.

In [ ]:
import os, sys, json, random, time, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'sentence-transformers', 'scikit-learn', 'pydantic', 'matplotlib'], check=False)
    if not Path('/content/c4/agents').exists():
        from google.colab import files
        uploaded = files.upload()                       # choose colab_bundle.zip
        import zipfile; zipfile.ZipFile(next(iter(uploaded))).extractall('/content/c4')
    ROOT = Path('/content/c4')
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / 'agents').is_dir() and ROOT != ROOT.parent:
        ROOT = ROOT.parent                              # find the project folder from wherever the notebook runs
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
if not IN_COLAB: os.environ.setdefault('HF_HUB_OFFLINE', '1')     # on the Mac the models come from this computer, nothing is downloaded
import numpy as np, torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('project folder:', ROOT); print('device:', DEVICE, '| on Colab:', IN_COLAB)

## Plot style

One small cell that makes every graph below look the same (clean grid, no top/right border) and gives each method its own colour, so a method has the same colour in every graph.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
COLORS = {'TF-IDF': '#9aa5b1', 'MiniLM': '#4c78a8', 'CodeBERT (not trained)': '#c9a227', 'CodeBERT (fine-tuned)': '#d1495b',
          'TF-IDF + MiniLM': '#2f6f4e', 'TF-IDF + CodeBERT (fine-tuned)': '#9b5de5', 'TF-IDF + MiniLM + CodeBERT': '#e07a1f'}
def color(name):
    return COLORS.get(name, '#444444')

def show_table(rows, columns, width=44):
    """Print {name: {metric: value}} as an aligned text table."""
    print(f"{'':{width}s}" + ''.join(f'{c:>10s}' for c in columns))
    for name, r in rows.items():
        print(f'{name:{width}s}' + ''.join(f'{r[c]:10.3f}' for c in columns))

## 2. Settings

* `CODEBERT_DIR`: the fine-tuned model to test (default: the one the app uses).
* `PROJECTS`: which real projects to test. `CHUNK_WORDS`: stage 1 scores a file by its best chunk of this many words (the project setting).

In [ ]:
PROJECTS = ['itrust', 'etour', 'smos', 'eanci', 'libest', 'cm1', 'ebt']
CODEBERT_DIR = ROOT / 'models' / 'c4-codebert'
SEED = 13
assert CODEBERT_DIR.is_dir(), f'{CODEBERT_DIR} not found: put the fine-tuned model folder there (see the training notebook)'
print('model under test:', CODEBERT_DIR)

## 3. Load the projects and the three embedding signals

For every project and every requirement the stage-1 score of each file is computed with the same functions the pipeline uses:
* **TF-IDF**: word overlap (rare words count more).
* **MiniLM**: a general sentence-embedding model.
* **CodeBERT**: the fine-tuned model.
* Fusions add the signals' z-scores per requirement (each signal is first rescaled so that they are comparable).

In [ ]:
import json
from shared.schemas.traceability import Artifact
from shared.embeddings.embedder import Embedder
from agents.traceability_agent.preprocess import retrieval_text
from agents.traceability_agent.retrieval import embedding_matrix, tfidf_matrix, zscore_fuse
from agents.traceability_agent.config import CHUNK_WORDS
PROCESSED = ROOT / 'data' / 'processed'

def load(name):
    """Requirements, code files and the expert-labelled links (as index sets) of one project."""
    arts = [Artifact(**a) for a in json.loads((PROCESSED / name / 'artifacts.json').read_text(encoding='utf-8'))]
    traces = json.loads((PROCESSED / name / 'traces.json').read_text(encoding='utf-8'))
    reqs, code = [a for a in arts if a.type == 'requirement'], [a for a in arts if a.type != 'requirement']
    r_idx, c_idx = {a.id: i for i, a in enumerate(reqs)}, {a.id: i for i, a in enumerate(code)}
    gold = {}
    for t in traces:
        if t['source_id'] in r_idx and t['target_id'] in c_idx: gold.setdefault(r_idx[t['source_id']], set()).add(c_idx[t['target_id']])
    return reqs, code, gold

mini, bert = Embedder('all-MiniLM-L6-v2'), Embedder(str(CODEBERT_DIR))
def signal_matrices(reqs, code):
    tf, e_mini = tfidf_matrix(reqs, code, retrieval_text, CHUNK_WORDS), embedding_matrix(reqs, code, mini, retrieval_text, CHUNK_WORDS)
    e_bert = embedding_matrix(reqs, code, bert, retrieval_text, CHUNK_WORDS)
    return {'TF-IDF': tf, 'MiniLM': e_mini, 'CodeBERT': e_bert, 'TF-IDF + MiniLM (before)': zscore_fuse([tf, e_mini]),
            'TF-IDF + CodeBERT': zscore_fuse([tf, e_bert]), 'MiniLM + CodeBERT': zscore_fuse([e_mini, e_bert]), 'TF-IDF + MiniLM + CodeBERT (after)': zscore_fuse([tf, e_mini, e_bert])}

## 4. Metrics

* **recall@k**: of all true links of the project, the share that is in the top k files of its requirement (pooled).
* **hit@k**: the share of requirements with at least one true link in the top k.
* **MRR**: the mean of 1 / rank of the first true link (1.0 = always first).
* **Bootstrap** (next section): resample the requirements 2000 times to see whether a difference could be luck.

In [ ]:
KS = (1, 3, 5, 10, 20)
def ranks_of_truth(matrix, gold):
    out = {}
    for i, truth in gold.items():
        position = {int(c): r + 1 for r, c in enumerate(np.argsort(-matrix[i], kind='stable'))}; out[i] = sorted(position[t] for t in truth)
    return out
def summarize(ranks):
    total = sum(len(r) for r in ranks.values())
    out = {f'recall@{k}': sum(sum(x <= k for x in r) for r in ranks.values()) / total for k in KS}
    out.update({f'hit@{k}': float(np.mean([r[0] <= k for r in ranks.values()])) for k in (1, 3, 5, 10)}); out['MRR'] = float(np.mean([1 / r[0] for r in ranks.values()])); return out
def bootstrap(a, b, k, n=2000):
    ids = sorted(a); diff = np.array([sum(x <= k for x in b[i]) - sum(x <= k for x in a[i]) for i in ids], dtype=float); rng = np.random.default_rng(SEED)
    means = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(n)]
    return float(diff.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

results, t0 = {}, time.time()
for p in PROJECTS:
    reqs, code, gold = load(p); ranks = {m: ranks_of_truth(x, gold) for m, x in signal_matrices(reqs, code).items()}
    results[p] = {'table': {m: summarize(r) for m, r in ranks.items()}, 'ranks': ranks, 'true_links': sum(len(v) for v in gold.values()), 'requirements': len(gold)}
    print(f'{p:7s} done ({time.time() - t0:.0f} s)')

## 5. Results on the real projects

### Table per project

In [ ]:
cols = ['recall@5', 'recall@10', 'recall@20', 'hit@3', 'MRR']
for p, r in results.items():
    print(f"\n{p}  ({r['requirements']} requirements with links, {r['true_links']} true links)"); show_table(r['table'], cols)
methods = list(next(iter(results.values()))['table'])
mean = {m: {c: float(np.mean([r['table'][m][c] for r in results.values()])) for c in cols} for m in methods}
print('\naverage over the projects (each project counts once)'); show_table(mean, cols)

### Graph 1: recall by depth, project by project

One small graph per project: how many of the true links are found when you look at the top 1, 3, 5, 10 and 20 files. The bold orange line is the three-signal pipeline now in the app; the green line is the pipeline before. Lines that are higher find the true links earlier.

In [ ]:
ps = list(results); cols_n = 4; rows_n = int(np.ceil(len(ps) / cols_n))
fig, axes = plt.subplots(rows_n, cols_n, figsize=(14, 3.1 * rows_n), squeeze=False)
for ax, p in zip(axes.ravel(), ps):
    for m in ('TF-IDF', 'MiniLM', 'CodeBERT', 'TF-IDF + MiniLM (before)', 'TF-IDF + MiniLM + CodeBERT (after)'):
        nm = 'TF-IDF + MiniLM' if 'before' in m else 'TF-IDF + MiniLM + CodeBERT' if 'after' in m else m
        ax.plot(KS, [results[p]['table'][m][f'recall@{k}'] for k in KS], marker='o', ms=3, color=color(nm), lw=2.4 if 'after' in m else 1.2, label=m)
    ax.set_xscale('log'); ax.set_xticks(KS); ax.set_xticklabels([str(k) for k in KS]); ax.set_ylim(0, 1.02); ax.set_title(p)
for ax in axes.ravel()[len(ps):]: ax.axis('off')
axes[0][0].legend(fontsize=6, loc='lower right'); fig.supxlabel('k (files looked at per requirement)'); fig.supylabel('recall@k'); plt.tight_layout(); plt.show()

### Graph 2: before and after at recall@10

In [ ]:
x = np.arange(len(ps)); w = 0.38; fig, ax = plt.subplots(figsize=(9, 3.6))
b = [results[p]['table']['TF-IDF + MiniLM (before)']['recall@10'] for p in ps]; a = [results[p]['table']['TF-IDF + MiniLM + CodeBERT (after)']['recall@10'] for p in ps]
ax.bar(x - w / 2, b, w, color=color('TF-IDF + MiniLM'), label='before: TF-IDF + MiniLM'); ax.bar(x + w / 2, a, w, color=color('TF-IDF + MiniLM + CodeBERT'), label='after: + CodeBERT')
for xi, (bv, av) in enumerate(zip(b, a)): ax.text(xi + w / 2, av + .01, f'{av - bv:+.2f}', ha='center', fontsize=8, color='#2f6f4e' if av >= bv else '#b3261e')
ax.set_xticks(x); ax.set_xticklabels(ps); ax.set_ylim(0, 1.1); ax.set_ylabel('recall@10'); ax.set_title('recall@10 per project (number = change)'); ax.legend(loc='lower left', fontsize=8); plt.tight_layout(); plt.show()

### Graph 3: is the difference real? (bootstrap)

How many extra true links per requirement the three-signal pipeline finds in the top 5 (left) and top 10 (right), with a 95% interval. **Green** = the whole interval is above 0 (a supported gain). **Grey** = the interval contains 0 (cannot be told from luck). **Red** = a supported loss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True); boot = {}
for ax, k in zip(axes, (5, 10)):
    for i, p in enumerate(ps):
        m, lo, hi = bootstrap(results[p]['ranks']['TF-IDF + MiniLM (before)'], results[p]['ranks']['TF-IDF + MiniLM + CodeBERT (after)'], k); boot[(p, k)] = (m, lo, hi)
        ax.errorbar(m, i, xerr=[[m - lo], [hi - m]], fmt='o', capsize=3, color='#2f6f4e' if lo > 0 else '#b3261e' if hi < 0 else '#6b7280')
    ax.axvline(0, color='black', lw=1); ax.set_title(f'top {k}'); ax.set_xlabel('extra true links per requirement (after minus before)')
axes[0].set_yticks(range(len(ps))); axes[0].set_yticklabels(ps); axes[0].invert_yaxis(); plt.tight_layout(); plt.show()

## 6. Spring PetClinic

PetClinic is a real application (37 files). Its 26 requirements were written from the code, and the answer key (`apps/seed/petclinic_gold.py`) was made by reading the code, so it is **not expert-labelled**: it is good for comparing methods with each other, not for absolute accuracy. 22 requirements have code; 4 do not and are left out. This section needs the clone in `data/seed/spring-petclinic` and is skipped if it is missing.

In [ ]:
from apps.seed import petclinic as pc, petclinic_gold
pet = {}
if (pc.DEFAULT_REPO / 'src' / 'main').is_dir():
    files = pc.scan(pc.DEFAULT_REPO); reqs_p = pc.build_requirements()['items']
    targets = [Artifact(id=f.id, type='code', text=f.text) for f in files]; sources = [Artifact(id=r['id'], type='requirement', text=r['text']) for r in reqs_p]
    gold_p = petclinic_gold.gold_pairs(); tid = {t.id: j for j, t in enumerate(targets)}
    gold_idx = {i: {tid[g] for g in gold_p[s.id]} for i, s in enumerate(sources) if s.id in gold_p}
    ranks_p = {m: ranks_of_truth(x, gold_idx) for m, x in signal_matrices(sources, targets).items()}
    pet = {m: summarize(r) for m, r in ranks_p.items()}
    print(f'{len(gold_idx)} requirements with an answer key, {sum(len(v) for v in gold_idx.values())} true links'); show_table(pet, cols)
    fig, ax = plt.subplots(figsize=(8, 3.2)); names = list(pet); ax.barh(range(len(names)), [pet[m]['recall@5'] for m in names], color='#4c78a8')
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8); ax.invert_yaxis(); ax.set_xlabel('recall@5'); ax.set_title('PetClinic: recall@5 by method'); ax.set_xlim(0, 1); plt.tight_layout(); plt.show()
else:
    print('PetClinic is not cloned (data/seed/spring-petclinic): this section is skipped.')

## 7. Verdict (written by the notebook from its own numbers)

In [ ]:
b, a = mean['TF-IDF + MiniLM (before)'], mean['TF-IDF + MiniLM + CodeBERT (after)']
wins = [p for p in ps if results[p]['table']['TF-IDF + MiniLM + CodeBERT (after)']['recall@10'] > results[p]['table']['TF-IDF + MiniLM (before)']['recall@10']]
losses = [p for p in ps if results[p]['table']['TF-IDF + MiniLM + CodeBERT (after)']['recall@10'] < results[p]['table']['TF-IDF + MiniLM (before)']['recall@10']]
supported = [p for p in ps if boot[(p, 10)][1] > 0 or boot[(p, 5)][1] > 0]
print(f"average recall@10: before {b['recall@10']:.3f} -> after {a['recall@10']:.3f} | average MRR: {b['MRR']:.3f} -> {a['MRR']:.3f}")
print(f'better in {len(wins)} of {len(ps)} projects {wins}; worse in {len(losses)} {losses}; supported gain (top 5 or 10) in {len(supported)} {supported}')
if len(supported) > len(ps) // 2: print('CONCLUSION: the three-signal pipeline gives a supported improvement.')
elif a['recall@10'] >= b['recall@10']: print('CONCLUSION: no loss on average and a small possible gain, but it cannot be told from luck on most projects.')
else: print('CONCLUSION: adding CodeBERT does not improve stage 1 on these real projects.')

## 8. Save the results

In [ ]:
res = ROOT / 'data' / 'results' / 'stage1_codebert_real_projects.json'
res.write_text(json.dumps({'model': CODEBERT_DIR.name, 'chunk_words': CHUNK_WORDS, 'projects': {p: {'requirements': r['requirements'], 'true_links': r['true_links'], 'table': r['table']} for p, r in results.items()},
                           'mean_over_projects': mean, 'petclinic': pet}, indent=1))
print('saved', res)

## What to say about this notebook

* The pipeline *before* is TF-IDF + MiniLM, *after* adds the fine-tuned CodeBERT as a third signal, tested on seven real expert-labelled projects the model never saw, plus PetClinic.
* Differences are shown per project with a bootstrap interval, so a real effect can be told from luck.
* The graphs and the verdict are produced by the notebook from its own run.